# Lagrangian Dynamics for Serial Robots

**Primary Reference:** Spong, Hutchinson & Vidyasagar, *Robot Modeling and Control*, Ch. 7  
**Supporting References:**  
- Lynch & Park, *Modern Robotics*, Ch. 8 — Dynamics  
- Craig, *Introduction to Robotics*, Ch. 6  
- MIT OCW 6.832 Underactuated Robotics — https://underactuated.mit.edu  

---

## The Lagrangian

You already know:

$$
\mathcal{L} = T - V
$$

Where $T$ is the total kinetic energy of the robot and $V$ is the total potential energy. Both are summed over all $n$ links.

The Lagrangian is a function of joint angles $\boldsymbol{\theta}$ and joint velocities $\dot{\boldsymbol{\theta}}$.

---

## Kinetic Energy

Each link has two contributions to kinetic energy — translational and rotational:

$$
T_i = \frac{1}{2} m_i \, \dot{\mathbf{r}}_{c_i}^T \dot{\mathbf{r}}_{c_i} + \frac{1}{2} \boldsymbol{\omega}_i^T \, \mathcal{I}_i \, \boldsymbol{\omega}_i
$$

Where:
- $m_i$ — mass of link $i$
- $\dot{\mathbf{r}}_{c_i}$ — velocity of the center of mass of link $i$ (3x1)
- $\boldsymbol{\omega}_i$ — angular velocity of link $i$ (3x1)
- $\mathcal{I}_i$ — inertia tensor of link $i$ about its center of mass (3x3)

### Connecting to the Jacobian

The center of mass velocity and angular velocity can both be expressed in terms of joint velocities $\dot{\boldsymbol{\theta}}$ using a **link Jacobian** $J^{(i)}$:

$$
\begin{bmatrix} \dot{\mathbf{r}}_{c_i} \\ \boldsymbol{\omega}_i \end{bmatrix} = J^{(i)}(\boldsymbol{\theta}) \, \dot{\boldsymbol{\theta}}
$$

$J^{(i)}$ is the Jacobian evaluated at the center of mass of link $i$, not the end-effector. It is built the same way as the end-effector Jacobian but using the center of mass position as $p_n$.

Substituting:

$$
T_i = \frac{1}{2} \dot{\boldsymbol{\theta}}^T \left[ m_i J_v^{(i)T} J_v^{(i)} + J_\omega^{(i)T} \mathcal{I}_i J_\omega^{(i)} \right] \dot{\boldsymbol{\theta}}
$$

Where $J_v^{(i)}$ is the top 3 rows (linear) and $J_\omega^{(i)}$ is the bottom 3 rows (angular) of $J^{(i)}$.

### The Mass Matrix

Summing over all links, the total kinetic energy is:

$$
T = \frac{1}{2} \dot{\boldsymbol{\theta}}^T M(\boldsymbol{\theta}) \, \dot{\boldsymbol{\theta}}
$$

Where the **mass matrix** (also called the inertia matrix) is:

$$
M(\boldsymbol{\theta}) = \sum_{i=1}^{n} \left[ m_i J_v^{(i)T} J_v^{(i)} + J_\omega^{(i)T} \mathcal{I}_i J_\omega^{(i)} \right]
$$

$M$ is an $n \times n$ symmetric positive-definite matrix. It depends on $\boldsymbol{\theta}$, meaning the effective inertia of the robot changes with configuration.

**Reference:** Spong Ch. 7.3

---

## Potential Energy

The potential energy of each link is due to gravity acting on its center of mass:

$$
V_i = m_i \, \mathbf{g}^T \mathbf{r}_{c_i}(\boldsymbol{\theta})
$$

Where:
- $\mathbf{g}$ — gravity vector, typically $[0, 0, -9.81]^T$ in the base frame (3x1)
- $\mathbf{r}_{c_i}$ — position of the center of mass of link $i$, expressed in the base frame (3x1)

$\mathbf{r}_{c_i}$ is extracted from the forward kinematics T matrix up to link $i$, evaluated at the center of mass offset.

Total potential energy:

$$
V = \sum_{i=1}^{n} m_i \, \mathbf{g}^T \mathbf{r}_{c_i}(\boldsymbol{\theta})
$$

---

## The Euler-Lagrange Equations

The equations of motion for joint $i$ are given by:

$$
\frac{d}{dt}\frac{\partial \mathcal{L}}{\partial \dot{\theta}_i} - \frac{\partial \mathcal{L}}{\partial \theta_i} = \tau_i
$$

Where $\tau_i$ is the torque (or force, for prismatic joints) applied at joint $i$.

Applying this to $\mathcal{L} = T - V$ and expanding the derivatives produces the **standard form** of the robot equations of motion:

$$
M(\boldsymbol{\theta})\ddot{\boldsymbol{\theta}} + C(\boldsymbol{\theta}, \dot{\boldsymbol{\theta}})\dot{\boldsymbol{\theta}} + \mathbf{g}(\boldsymbol{\theta}) = \boldsymbol{\tau}
$$

Where:
- $M(\boldsymbol{\theta})$ — mass matrix ($n \times n$)
- $C(\boldsymbol{\theta}, \dot{\boldsymbol{\theta}})$ — Coriolis and centripetal matrix ($n \times n$)
- $\mathbf{g}(\boldsymbol{\theta})$ — gravity vector ($n \times 1$)
- $\boldsymbol{\tau}$ — joint torques/forces ($n \times 1$)

**Reference:** Spong Ch. 7.4, Lynch & Park Ch. 8.1

---

## The Coriolis Matrix

$C$ is derived from $M$ using the **Christoffel symbols of the first kind**:

$$
c_{ijk} = \frac{1}{2}\left(\frac{\partial m_{ij}}{\partial \theta_k} + \frac{\partial m_{ik}}{\partial \theta_j} - \frac{\partial m_{jk}}{\partial \theta_i}\right)
$$

Each element of $C$ is:

$$
C_{ij} = \sum_{k=1}^{n} c_{ijk} \, \dot{\theta}_k
$$

In practice with sympy, $C$ is computed by differentiating $M$ symbolically with respect to $\boldsymbol{\theta}$.

**Reference:** Spong Ch. 7.4

---

## The Gravity Vector

$$
g_i(\boldsymbol{\theta}) = \frac{\partial V}{\partial \theta_i} = \sum_{j=i}^{n} m_j \, \mathbf{g}^T \frac{\partial \mathbf{r}_{c_j}}{\partial \theta_i}
$$

With sympy, this is simply `smp.diff(V, theta_i)` for each joint.

---

## What the YAML Needs

To compute $T$ and $V$, each link needs additional physical properties beyond the DH parameters:

| Parameter | Symbol | Description |
|---|---|---|
| `mass` | $m_i$ | Link mass (kg) |
| `center_of_mass` | $\mathbf{r}_{c_i}$ | CoM offset from joint frame origin (x, y, z in meters) |
| `inertia` | $\mathcal{I}_i$ | 3x3 inertia tensor about CoM (kg·m²) |

These should be added to each joint entry in the YAML.

---

## What to Implement

1. **Update YAML** — add `mass`, `center_of_mass`, and `inertia` to each joint
2. **`jacobian.py`** — add a function to build the link Jacobian $J^{(i)}$ evaluated at the center of mass of each link
3. **`dynamics.py`** (new file) — compute $M$, $C$, and $\mathbf{g}$ symbolically using sympy
4. The controller passes symbolic joint variables; dynamics returns the full equations of motion

With sympy, $C$ and $\mathbf{g}$ are computed by differentiating $M$ and $V$ — no need to implement the Christoffel symbol formula by hand.

## Example — Mass Matrix for One Link

In [ ]:
import sympy as smp

# Suppose J_v and J_w are the linear and angular rows of the link Jacobian
# for link i (each is a 3xn sympy Matrix), m is the link mass (scalar),
# and I is the 3x3 inertia tensor (sympy Matrix).

# Contribution of link i to the mass matrix:
# M_i = m * J_v.T * J_v + J_w.T * I * J_w

# Sum M_i over all links to get the full M(theta).

## Example — Coriolis and Gravity via Sympy Differentiation

In [ ]:
import sympy as smp

# Define symbolic joint variables
# theta = smp.Matrix(smp.symbols('theta1:n+1'))  # n joint symbols
# theta_dot = smp.Matrix(smp.symbols('dtheta1:n+1'))

# Gravity vector in base frame (pointing down)
# g_vec = smp.Matrix([0, 0, -9.81])

# Potential energy (sum over links)
# V = sum(m_i * g_vec.dot(r_ci) for each link i)

# Gravity vector g(theta) — one element per joint
# g_vec_joints = smp.Matrix([smp.diff(V, theta[i]) for i in range(n)])

# Coriolis matrix C — uses Christoffel symbols derived from M
# C[i,j] = sum_k( 0.5*(diff(M[i,j], theta[k]) + diff(M[i,k], theta[j]) - diff(M[j,k], theta[i])) * theta_dot[k] )
# In practice: iterate over i, j, k with smp.diff